# Smartphones Domain

This notebook builds the **smartphones** slice of the Wikidata multihop benchmark. It queries Wikidata for smartphone models grouped by manufacturer, operating system, and release year, then generates multi-hop retrieval examples across five complexity levels (L1–L5).

**Produces:**
- `BenchmarkExample` objects for the smartphones domain (L1–L5), consumed by the main dataset assembly notebook
- Candidate pools: manufacturer lists, manufacturer × year, manufacturer × OS, manufacturer × OS × year, and reference-model comparison sets
- Query generators: `generate_smartphones_example` and `generate_smartphones_example_advanced`

## 1. Configuration

Load shared benchmark helpers. When this notebook is run standalone the helpers are imported from `common_helpers.py`; when executed from the orchestrator notebook the globals are already populated.

In [ ]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires the nbformat package.
from pathlib import Path
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())


## 2. Imports

In [ ]:
import pandas as pd
import random
import datetime as dt
from collections import defaultdict
from typing import Optional, List, Tuple, Dict, Any

Q_SMARTPHONE = ensure_qid("смартфон", fallback_qid="Q22645")            # smartphone
Q_MOBILE_PHONE = ensure_qid("мобильный телефон", fallback_qid="Q17517")  # mobile phone
Q_ANDROID = ensure_qid("Android", fallback_qid="Q94")
Q_IOS = ensure_qid("iOS", fallback_qid="Q48493")

Q_SMARTPHONE_MODEL = resolve_qid("модель смартфона", "ru") or resolve_qid("smartphone model", "en")
Q_MOBILE_PHONE_MODEL = resolve_qid("модель мобильного телефона", "ru") or resolve_qid("mobile phone model", "en")

SMARTPHONE_TYPE_QIDS: List[str] = [q for q in [
    Q_SMARTPHONE,
    Q_MOBILE_PHONE,
    Q_SMARTPHONE_MODEL,
    Q_MOBILE_PHONE_MODEL,
] if q]

SMARTPHONE_MAKER_ANCHORS = [
    ("Q312", "Apple"),
    ("Q27414", "Samsung"),
    ("Q174187", "Xiaomi"),
    ("Q175751", "Huawei"),
    ("Q95", "Google"),
    ("Q318144", "Sony"),
    ("Q3884", "Nokia"),
    ("Q207194", "Motorola"),
    ("Q215380", "OnePlus"),
    ("Q1134006", "Honor"),
    ("Q1078460", "Oppo"),
    ("Q679694", "Vivo"),
    ("Q257998", "LG"),
    ("Q15148", "HTC"),
    ("Q192608", "Lenovo"),
]
SMARTPHONE_OS_ANCHORS = [
    (Q_ANDROID, "Android"),
    (Q_IOS, "iOS"),
]

## 3. In-memory caches

In [ ]:
_LABEL_CACHE: Dict[str, str] = {}
_SMARTPHONE_ENUM_CACHE: Dict[str, List[Any]] = {}
_SMARTPHONE_ENUM_CURSOR: Dict[str, int] = defaultdict(int)
_SMARTPHONE_CANDIDATE_CACHE: Dict[Tuple[Any, ...], List[Any]] = {}

## 4. SPARQL / Wikidata helpers

Label lookup, SPARQL query fragment builders, and small utility functions shared by all pool-building queries.

In [ ]:
def _norm_lbl(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def get_label_ru_en_cached(qid: str) -> str:
    qid = (qid or "").strip()
    if not qid or not qid.startswith("Q"):
        return ""
    if qid in _LABEL_CACHE:
        return _LABEL_CACHE[qid]

    query = f"""
SELECT ?ru ?en WHERE {{
  OPTIONAL {{ wd:{qid} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ wd:{qid} rdfs:label ?en FILTER(LANG(?en)='en') }}
}}
LIMIT 1
"""
    lbl = ""
    try:
        rows = rows_from_select(wd.sparql_select(query))
        if rows:
            lbl = _norm_lbl(rows[0].get("ru")) or _norm_lbl(rows[0].get("en"))
    except Exception:
        lbl = ""
    _LABEL_CACHE[qid] = lbl
    return lbl

def _smartphone_type_block(item_var: str = "p") -> str:
    values = " ".join(f"wd:{q}" for q in SMARTPHONE_TYPE_QIDS if q)
    return f"""
  ?{item_var} wdt:P31/wdt:P279* ?phoneClass .
  VALUES ?phoneClass {{ {values} }}
  OPTIONAL {{ ?{item_var} wdt:P306 ?_anyOs . }}
  FILTER(
    ?phoneClass = wd:{Q_SMARTPHONE}
    {"|| ?phoneClass = wd:" + Q_SMARTPHONE_MODEL if Q_SMARTPHONE_MODEL else ""}
    || BOUND(?_anyOs)
  )
"""

def _smartphone_date_block(item_var: str = "p") -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?d1 . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?d2 . }}",
        "BIND(COALESCE(?d1, ?d2) AS ?d)",
        "FILTER(BOUND(?d))",
    ]

def _smartphone_label_block(item_var: str = "p") -> str:
    return f"""
  OPTIONAL {{ ?{item_var} rdfs:label ?ru FILTER(LANG(?ru)='ru') }}
  OPTIONAL {{ ?{item_var} rdfs:label ?en FILTER(LANG(?en)='en') }}
  BIND(COALESCE(?ru, ?en) AS ?label)
"""

def _smartphone_gold_limit(k: int, complexity: str) -> int:
    if complexity in ("L4", "L5"):
        return max(50, k * 20)
    return max(80, k * 25)

def _smartphone_items_from_rows(rows: List[Dict[str, Any]], q_key: str = "p", lbl_key: str = "label") -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get(q_key, ""))
        lbl = _norm_lbl(r.get(lbl_key))
        if not qid or qid in seen:
            continue
        seen.add(qid)
        if not lbl:
            lbl = get_label_ru_en_cached(qid)
        if qid:
            out.append((qid, lbl))
    return out

## 5. Pool building — manufacturer candidates

Functions to query Wikidata for valid manufacturers and enumerate them during generation, with hard-coded anchor fallbacks in case the SPARQL endpoint returns no results.

In [ ]:
def run_smartphones_query(
    maker_qid: Optional[str] = None,
    os_qid: Optional[str] = None,
    year_from: Optional[int] = None,
    year_to: Optional[int] = None,
    not_maker_qid: Optional[str] = None,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        "?p wdt:P176 ?maker .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
    ]

    if maker_qid:
        where_lines.append(f"?p wdt:P176 wd:{maker_qid} .")
    if os_qid:
        where_lines.append(f"?p wdt:P306 wd:{os_qid} .")
    if not_maker_qid:
        where_lines.append(f"FILTER NOT EXISTS {{ ?p wdt:P176 wd:{not_maker_qid} . }}")

    if year_from is not None or year_to is not None:
        where_lines.extend(_smartphone_date_block("p"))
        if year_from is not None:
            where_lines.append(f"FILTER(YEAR(?d) >= {int(year_from)})")
        if year_to is not None:
            where_lines.append(f"FILTER(YEAR(?d) <= {int(year_to)})")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines

def _next_candidate(key: str, builder, rng: random.Random):
    if key not in _SMARTPHONE_ENUM_CACHE:
        items = list(builder() or [])
        rng.shuffle(items)
        _SMARTPHONE_ENUM_CACHE[key] = items
        _SMARTPHONE_ENUM_CURSOR[key] = 0

    items = _SMARTPHONE_ENUM_CACHE[key]
    if not items:
        return None

    cur = _SMARTPHONE_ENUM_CURSOR.get(key, 0)
    if cur >= len(items):
        if len(items) > 1:
            rng.shuffle(items)
        cur = 0
    _SMARTPHONE_ENUM_CURSOR[key] = cur + 1
    return items[cur]

def _fallback_maker_candidates() -> List[Tuple[str, str, int]]:
    out = []
    for qid, lbl in SMARTPHONE_MAKER_ANCHORS:
        out.append((qid, lbl or get_label_ru_en_cached(qid) or "", 999))
    return out

def _fallback_maker_os_candidates() -> List[Tuple[str, str, str, str, int]]:
    out = []
    for maker_qid, maker_lbl in SMARTPHONE_MAKER_ANCHORS:
        for os_qid, os_lbl in SMARTPHONE_OS_ANCHORS:
            out.append((maker_qid, maker_lbl or get_label_ru_en_cached(maker_qid) or "",
                        os_qid, os_lbl or get_label_ru_en_cached(os_qid) or "", 999))
    return out

def find_smartphone_maker_candidates(min_models: int = 5, limit: int = 80) -> List[Tuple[str, str, int]]:
    key = ("makers", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and maker_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, cnt))

    if not out:
        out = _fallback_maker_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

## 6. Pool building — manufacturer × year and manufacturer × OS candidates

Aggregation queries that combine manufacturer constraints with release-year windows and operating-system filters. These pools back the L2–L3 (year-bounded) and the L5 (negated-manufacturer) template families.

In [ ]:
def find_smartphone_maker_year_candidates(
    min_models: int = 5,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 240,
) -> List[Tuple[str, str, int, int]]:
    key = ("maker_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and maker_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, yy, cnt))

    if not out:
        years = list(range(2018, 2025))
        makers = find_smartphone_maker_candidates(min_models=max(2, min_models - 2), limit=20)
        for maker_qid, maker_lbl, _ in makers:
            for yy in years:
                out.append((maker_qid, maker_lbl, yy, max(min_models, 5)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_candidates(min_models: int = 5, limit: int = 160) -> List[Tuple[str, str, str, str, int]]:
    key = ("maker_os", int(min_models), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            cnt = 0
        if maker_qid and os_qid and maker_lbl and os_lbl and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, cnt))

    if not out:
        out = _fallback_maker_os_candidates()

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def find_smartphone_maker_os_year_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    limit: int = 320,
) -> List[Tuple[str, str, str, str, int, int]]:
    key = ("maker_os_year", int(min_models), int(year_from), int(year_to), int(limit))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    sparql = f"""
SELECT ?maker ?makerLabel ?os ?osLabel ?yy (COUNT(DISTINCT ?p) AS ?cnt) WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 ?maker ;
     wdt:P306 ?os .
  {' '.join(_smartphone_date_block("p"))}
  BIND(YEAR(?d) AS ?yy)
  FILTER(?yy >= {int(year_from)} && ?yy <= {int(year_to)})
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}}
GROUP BY ?maker ?makerLabel ?os ?osLabel ?yy
HAVING(COUNT(DISTINCT ?p) >= {int(min_models)})
ORDER BY DESC(COUNT(DISTINCT ?p))
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    out: List[Tuple[str, str, str, str, int, int]] = []
    for r in rows:
        maker_qid = uri_to_qid(r.get("maker", ""))
        os_qid = uri_to_qid(r.get("os", ""))
        maker_lbl = _norm_lbl(r.get("makerLabel")) or (get_label_ru_en_cached(maker_qid) if maker_qid else "")
        os_lbl = _norm_lbl(r.get("osLabel")) or (get_label_ru_en_cached(os_qid) if os_qid else "")
        try:
            yy = int(float(r.get("yy", "0")))
            cnt = int(float(r.get("cnt", "0")))
        except Exception:
            yy, cnt = 0, 0
        if maker_qid and os_qid and maker_lbl and os_lbl and yy and cnt >= int(min_models):
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt))

    if not out:
        base = find_smartphone_maker_os_candidates(min_models=max(2, min_models), limit=40)
        for maker_qid, maker_lbl, os_qid, os_lbl, _ in base:
            for yy in range(2018, 2025):
                out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, max(min_models, 1)))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

def build_smartphone_l5_candidates(
    min_models: int = 1,
    year_from: int = 2010,
    year_to: int = 2025,
    base_limit: int = 120,
    not_makers_per_base: int = 4,
) -> List[Tuple[str, str, str, str, int, str, str, int]]:
    key = ("l5", int(min_models), int(year_from), int(year_to), int(base_limit), int(not_makers_per_base))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    base = find_smartphone_maker_os_year_candidates(
        min_models=min_models,
        year_from=year_from,
        year_to=year_to,
        limit=base_limit,
    )
    makers = find_smartphone_maker_candidates(min_models=max(2, min_models), limit=40)

    out: List[Tuple[str, str, str, str, int, str, str, int]] = []
    for maker_qid, maker_lbl, os_qid, os_lbl, yy, cnt in base:
        others = [(q, l) for q, l, _ in makers if q != maker_qid][:max(1, int(not_makers_per_base))]
        for not_maker_qid, not_maker_lbl in others:
            out.append((maker_qid, maker_lbl, os_qid, os_lbl, yy, not_maker_qid, not_maker_lbl, cnt))

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

## 7. Per-maker model fetching and advanced candidate builders

Fetches the full ordered chronology of models for a given manufacturer and derives year-window, threshold-year, and reference-model comparison candidate sets used by L2–L5 templates.

In [ ]:
_SMARTPHONE_MODELS_BY_MAKER_CACHE: Dict[Tuple[str, int, int, int], List[Dict[str, Any]]] = {}

def _smartphone_date_block_named(
    item_var: str = "p",
    date_var: str = "d",
    d1_var: str = "d1",
    d2_var: str = "d2",
) -> List[str]:
    return [
        f"OPTIONAL {{ ?{item_var} wdt:P577 ?{d1_var} . }}",
        f"OPTIONAL {{ ?{item_var} wdt:P571 ?{d2_var} . }}",
        f"BIND(COALESCE(?{d1_var}, ?{d2_var}) AS ?{date_var})",
        f"FILTER(BOUND(?{date_var}))",
    ]


def fetch_smartphone_models_for_maker(
    maker_qid: str,
    min_year: int = 2007,
    max_year: int = 2026,
    limit: int = 300,
) -> List[Dict[str, Any]]:
    key = (maker_qid, int(min_year), int(max_year), int(limit))
    if key in _SMARTPHONE_MODELS_BY_MAKER_CACHE:
        return _SMARTPHONE_MODELS_BY_MAKER_CACHE[key]

    sparql = f"""
SELECT DISTINCT ?p ?label ?d WHERE {{
  {_smartphone_type_block("p")}
  ?p wdt:P176 wd:{maker_qid} .
  ?p wikibase:sitelinks ?sl .
  FILTER(?sl >= 1)
  {' '.join(_smartphone_date_block_named("p", "d", "d1m", "d2m"))}
  FILTER(YEAR(?d) >= {int(min_year)} && YEAR(?d) <= {int(max_year)})
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?d) ?label
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))

    out: List[Dict[str, Any]] = []
    seen: set = set()
    for r in rows:
        qid = uri_to_qid(r.get("p", ""))
        if not qid or qid in seen:
            continue
        seen.add(qid)

        lbl = _norm_lbl(r.get("label")) or get_label_ru_en_cached(qid)
        d = _norm_lbl(r.get("d"))
        m = re.match(r"^(\d{4})", d)
        if not m:
            continue
        yy = int(m.group(1))
        if yy < int(min_year) or yy > int(max_year):
            continue
        out.append({
            "qid": qid,
            "label": lbl,
            "date": d,
            "year": yy,
        })

    _SMARTPHONE_MODELS_BY_MAKER_CACHE[key] = out
    return out


def find_smartphone_maker_year_window_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    min_span_years: int = 2,
    max_span_years: int = 4,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, int, int, int]]:
    key = (
        "maker_year_window",
        int(min_models),
        int(maker_limit),
        int(min_span_years),
        int(max_span_years),
        int(max_candidates_per_maker),
    )
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, int, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        if len(models) < min_models:
            continue

        year_counts: Dict[int, int] = defaultdict(int)
        for m in models:
            year_counts[int(m["year"])] += 1

        years = sorted(year_counts)
        local: List[Tuple[str, str, int, int, int]] = []
        for i, y1 in enumerate(years):
            total = 0
            for j in range(i, len(years)):
                y2 = years[j]
                total += int(year_counts[y2])
                span = y2 - y1 + 1
                if span < int(min_span_years):
                    continue
                if span > int(max_span_years):
                    break
                if total >= int(min_models):
                    local.append((maker_qid, maker_lbl, y1, y2, total))

        local.sort(key=lambda x: (abs((x[3] - x[2] + 1) - 3), abs(x[4] - 7), x[2], x[3]))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_maker_threshold_year_candidates(
    min_models: int = 5,
    maker_limit: int = 24,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, int, int]]:
    key = ("maker_threshold_year", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 1, 6), limit=maker_limit)
    out: List[Tuple[str, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        years = sorted(int(m["year"]) for m in models)
        uniq = sorted(set(years))
        if len(uniq) < 3:
            continue

        local: List[Tuple[str, str, str, int, int]] = []
        for yy in uniq[1:-1]:
            cnt_ge = sum(1 for y in years if y >= yy)
            cnt_lt = sum(1 for y in years if y < yy)
            cnt_le = sum(1 for y in years if y <= yy)
            cnt_gt = sum(1 for y in years if y > yy)

            if cnt_ge >= int(min_models) and cnt_lt >= 1:
                local.append((maker_qid, maker_lbl, "from", yy, cnt_ge))
            if cnt_le >= int(min_models) and cnt_gt >= 1:
                local.append((maker_qid, maker_lbl, "to", yy, cnt_le))

        local.sort(key=lambda x: (abs(x[4] - 8), x[3], 0 if x[2] == "from" else 1))
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_reference_comparison_candidates(
    min_models: int = 5,
    maker_limit: int = 20,
    max_candidates_per_maker: int = 12,
) -> List[Tuple[str, str, str, str, int, str, int]]:
    key = ("ref_compare", int(min_models), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models + 2, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models + 2:
            continue

        local: List[Tuple[str, str, str, str, int, str, int]] = []
        for i, m in enumerate(models):
            ref_qid = m["qid"]
            ref_lbl = _norm_lbl(m["label"])
            ref_year = int(m["year"])

            cnt_after = max(0, n - i - 1)
            cnt_before = max(0, i)

            if cnt_after >= int(min_models) and cnt_before >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "after", cnt_after))
            if cnt_before >= int(min_models) and cnt_after >= 1 and ref_lbl:
                local.append((maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, "before", cnt_before))

        local.sort(
            key=lambda x: (
                abs(x[6] - 8),
                abs(x[4] - 2018),
                0 if x[5] == "after" else 1,
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out


def find_smartphone_between_model_candidates(
    min_models_between: int = 2,
    maker_limit: int = 18,
    max_candidates_per_maker: int = 10,
) -> List[Tuple[str, str, str, str, int, str, str, int, int]]:
    key = ("between_models", int(min_models_between), int(maker_limit), int(max_candidates_per_maker))
    if key in _SMARTPHONE_CANDIDATE_CACHE:
        return _SMARTPHONE_CANDIDATE_CACHE[key]

    makers = find_smartphone_maker_candidates(min_models=max(min_models_between + 5, 7), limit=maker_limit)
    out: List[Tuple[str, str, str, str, int, str, str, int, int]] = []

    for maker_qid, maker_lbl, _ in makers:
        models = fetch_smartphone_models_for_maker(maker_qid)
        n = len(models)
        if n < min_models_between + 3:
            continue

        local: List[Tuple[str, str, str, str, int, str, str, int, int]] = []
        for i in range(n):
            left = models[i]
            for j in range(i + min_models_between + 1, n):
                right = models[j]
                between = j - i - 1
                if between < int(min_models_between):
                    continue

                year_gap = int(right["year"]) - int(left["year"])
                if year_gap < 1:
                    continue

                local.append((
                    maker_qid,
                    maker_lbl,
                    left["qid"],
                    _norm_lbl(left["label"]),
                    int(left["year"]),
                    right["qid"],
                    _norm_lbl(right["label"]),
                    int(right["year"]),
                    between,
                ))

        local.sort(
            key=lambda x: (
                abs(x[8] - 4),
                abs((x[7] - x[4]) - 4),
                x[4],
                x[7],
            )
        )
        out.extend(local[:max_candidates_per_maker])

    _SMARTPHONE_CANDIDATE_CACHE[key] = out
    return out

## 8. Query generators — relative-model SPARQL

SPARQL query builders for the L4 (before/after a reference model) and L5 (between two reference models) templates. These functions return the full query string, the gold answer list, and the parsed WHERE clauses.

In [ ]:
def run_smartphones_query_relative_model(
    maker_qid: str,
    ref_model_qid: str,
    relation: str = "after",
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    if relation not in {"after", "before"}:
        raise ValueError(f"Unsupported relation: {relation}")

    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?refModel {{ wd:{ref_model_qid} }}",
        *_smartphone_date_block_named("refModel", "refDate", "d1ref", "d2ref"),
        "FILTER(?p != ?refModel)",
    ]
    if relation == "after":
        where_lines.append("FILTER(?d > ?refDate)")
    else:
        where_lines.append("FILTER(?d < ?refDate)")

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines


def run_smartphones_query_between_models(
    maker_qid: str,
    left_model_qid: str,
    right_model_qid: str,
    limit: int = 120,
) -> Tuple[str, List[Tuple[str, str]], List[str]]:
    where_lines: List[str] = [
        _smartphone_type_block("p"),
        f"?p wdt:P176 wd:{maker_qid} .",
        "?p wikibase:sitelinks ?sl .",
        "FILTER(?sl >= 1)",
        *_smartphone_date_block_named("p", "d", "d1p", "d2p"),
        f"VALUES ?leftRef {{ wd:{left_model_qid} }}",
        *_smartphone_date_block_named("leftRef", "leftDate", "d1l", "d2l"),
        f"VALUES ?rightRef {{ wd:{right_model_qid} }}",
        *_smartphone_date_block_named("rightRef", "rightDate", "d1r", "d2r"),
        "FILTER(?p != ?leftRef && ?p != ?rightRef)",
        "FILTER(?leftDate < ?rightDate)",
        "FILTER(?d > ?leftDate && ?d < ?rightDate)",
    ]

    where = "\n  ".join(where_lines)
    sparql = f"""
SELECT DISTINCT ?p ?label WHERE {{
  {where}
  {_smartphone_label_block("p")}
}}
ORDER BY ASC(?label)
LIMIT {int(limit)}
"""
    rows = rows_from_select(wd.sparql_select(sparql))
    items = _smartphone_items_from_rows(rows, q_key="p", lbl_key="label")
    return sparql.strip(), items, where_lines

## 9. Dataset generation

Assembles `BenchmarkExample` records for each complexity level. `_generate_smartphones_example_default` dispatches to the appropriate candidate pool and query builder based on `complexity` (L1–L5). The public entry points `generate_smartphones_example` and `generate_smartphones_example_advanced` are imported by the orchestrator notebook.

In [ ]:
def _smartphones_now_iso() -> str:
    try:
        return utc_now_z()
    except Exception:
        return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")

def _smartphone_make_example(
    idx: int,
    complexity: str,
    query_text_ru: str,
    constraints: Dict[str, Any],
    items: List[Tuple[str, str]],
    sparql_query: str,
    template_id: str,
) -> BenchmarkExample:
    return BenchmarkExample(
        id=f"smartphones_{complexity.lower()}_{idx:05d}",
        domain="smartphones",
        complexity=complexity,
        query_text_ru=query_text_ru,
        constraints=constraints,
        requested_count=5,
        gold_answer_qids=[q for q, _ in items],
        gold_answer_labels_ru=[l for _, l in items],
        sparql_query=sparql_query,
        created_at=_smartphones_now_iso(),
        is_advanced=False,
        template_id=template_id,
        template_family="default",
        gold_truncated=False,
        ask_validator_sparql=None,
    )


def _generate_smartphones_example_default(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    k = 5

    if complexity == "L1":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L1",
                lambda: find_smartphone_maker_candidates(min_models=k, limit=120),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} моделей смартфонов производителя «{maker_lbl}».",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_only",
            )
        raise RuntimeError("Smartphones L1: no valid maker candidates")

    if complexity == "L2":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L2",
                lambda: find_smartphone_maker_year_window_candidates(
                    min_models=k,
                    maker_limit=24,
                    min_span_years=2,
                    max_span_years=4,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, y1, y2, _ = cand
            sparql, items, _ = run_smartphones_query(
                maker_qid=maker_qid,
                year_from=y1,
                year_to=y2,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных в период {y1}–{y2} годов.",
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": y1,
                    "year_to": y2,
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_year_window",
            )
        raise RuntimeError("Smartphones L2: no valid maker+year-window candidates")

    if complexity == "L3":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L3",
                lambda: find_smartphone_maker_threshold_year_candidates(
                    min_models=k,
                    maker_limit=24,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, mode, yy, _ = cand
            if mode == "from":
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_from=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не раньше {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_from": yy,
                    "year_mode": "from",
                }
                template_id = "smartphones_maker_year_from"
            else:
                sparql, items, _ = run_smartphones_query(
                    maker_qid=maker_qid,
                    year_to=yy,
                    limit=_smartphone_gold_limit(k, complexity),
                )
                query_text_ru = f"Назови {k} смартфонов производителя «{maker_lbl}», выпущенных не позже {yy} года."
                constraints = {
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "year_to": yy,
                    "year_mode": "to",
                }
                template_id = "smartphones_maker_year_to"

            if len(items) < k:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints=constraints,
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L3: no valid maker+threshold-year candidates")

    if complexity == "L4":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L4",
                lambda: find_smartphone_reference_comparison_candidates(
                    min_models=k,
                    maker_limit=20,
                    max_candidates_per_maker=12,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, ref_qid, ref_lbl, ref_year, relation, _ = cand
            sparql, items, _ = run_smartphones_query_relative_model(
                maker_qid=maker_qid,
                ref_model_qid=ref_qid,
                relation=relation,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < k:
                continue

            if relation == "after":
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_after_model"
            else:
                query_text_ru = (
                    f"Назови {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных раньше модели «{ref_lbl}» ({ref_year})."
                )
                template_id = "smartphones_maker_before_model"

            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=query_text_ru,
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "reference_model_qid": ref_qid,
                    "reference_model_label_ru": ref_lbl,
                    "reference_model_year": ref_year,
                    "relation": relation,
                },
                items=items,
                sparql_query=sparql,
                template_id=template_id,
            )
        raise RuntimeError("Smartphones L4: no valid maker+reference-model candidates")

    if complexity == "L5":
        for _ in range(max_attempts):
            cand = _next_candidate(
                "smartphones_L5",
                lambda: find_smartphone_between_model_candidates(
                    min_models_between=2,
                    maker_limit=18,
                    max_candidates_per_maker=10,
                ),
                rng,
            )
            if not cand:
                break
            maker_qid, maker_lbl, left_qid, left_lbl, left_year, right_qid, right_lbl, right_year, _ = cand
            sparql, items, _ = run_smartphones_query_between_models(
                maker_qid=maker_qid,
                left_model_qid=left_qid,
                right_model_qid=right_qid,
                limit=_smartphone_gold_limit(k, complexity),
            )
            if len(items) < 1:
                continue
            return _smartphone_make_example(
                idx=idx,
                complexity=complexity,
                query_text_ru=(
                    f"Подбери до {k} смартфонов производителя «{maker_lbl}», "
                    f"выпущенных позже модели «{left_lbl}» ({left_year}), "
                    f"но раньше модели «{right_lbl}» ({right_year})."
                ),
                constraints={
                    "maker_qid": maker_qid,
                    "maker_label_ru": maker_lbl,
                    "left_reference_model_qid": left_qid,
                    "left_reference_model_label_ru": left_lbl,
                    "left_reference_model_year": left_year,
                    "right_reference_model_qid": right_qid,
                    "right_reference_model_label_ru": right_lbl,
                    "right_reference_model_year": right_year,
                    "relation": "between_models",
                },
                items=items,
                sparql_query=sparql,
                template_id="smartphones_maker_between_models",
            )
        raise RuntimeError("Smartphones L5: no valid maker+between-models candidates")

    raise ValueError(f"Unknown complexity: {complexity}")


def generate_smartphones_example_advanced(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)

def generate_smartphones_example(
    complexity: str,
    idx: int,
    rng: random.Random,
    max_attempts: int = 40,
) -> BenchmarkExample:
    return _generate_smartphones_example_default(complexity, idx, rng, max_attempts=max_attempts)